In [1]:
!pip install numpy matplotlib ipywidgets


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3.11 -m pip install --upgrade pip


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
import ipywidgets as widgets
import os

# --- 1. PARAMETERS EXTRACTED FROM YOUR NRRD HEADER ---
file_path = 'path/to/raw/file'
data_type = np.uint8  # 'unsigned char'

# NRRD sizes are typically (X, Y, Z). 
# NumPy expects C-contiguous shapes as (Z, Y, X) for standard slicing.
dimensions = (509, 5500, 11000) # use correct dimensions



# --- 2. LOAD DATA VIA MEMORY MAPPING ---
if not os.path.exists(file_path):
    print(f"Error: File '{file_path}' not found in the current directory.")
else:
    print(f"Memory mapping {file_path}...")
    # np.memmap reads from disk on demand instead of loading into RAM
    volume = np.memmap(file_path, dtype=data_type, mode='r', shape=dimensions)
    print("Mapping successful!")

    # --- 3. CREATE THE INTERACTIVE VIEWER ---
    def view_slice(z):
        plt.figure(figsize=(8, 8))
        
        # Read ONLY slice 'z' from disk into RAM (~400MB)
        slice_data = volume[z, :, :]
        
        # DOWN-SAMPLE for plotting speed. 
        # Rendering a 20kx20k image in a notebook is too slow. 
        # 'step=10' means we only plot every 10th pixel (looks identical scaled down).
        step = 10 
        
        plt.imshow(slice_data[::step, ::step], cmap='gray')
        plt.title(f'Slice Z = {z}')
        plt.axis('off')
        plt.show()

    # Create an interactive slider for the Z-axis (0 to 814)
    interact(view_slice, z=widgets.IntSlider(min=0, max=dimensions[0]-1, step=1, value=dimensions[0]//2))

Memory mapping /home/tahmeed/nvIndexViewer/brainStem/fb76_4mpp.raw...
Mapping successful!


interactive(children=(IntSlider(value=254, description='z', max=508), Output()), _dom_classes=('widget-interac…

# 2nd Method

## need high end gpu and gui

In [3]:
!pip install numpy pyvista trame trame-vuetify trame-vtk

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3.11 -m pip install --upgrade pip


In [ ]:
import numpy as np
import pyvista as pv
import os

# --- 1. PARAMETERS ---
file_path = 'path/to/raw-volume/file'
data_type = np.uint8
dimensions = (815, 20000, 20000) # (Z, Y, X)

if not os.path.exists(file_path):
    print(f"Error: File '{file_path}' not found.")
else:
    print(f"Memory mapping {file_path}...")
    # Map the huge file (does not load into RAM yet)
    volume_map = np.memmap(file_path, dtype=data_type, mode='r', shape=dimensions)
    
    # --- 2. MASSIVE DOWNSAMPLING ---
    # We must shrink the data so it fits in RAM and the browser.
    # Taking every 100th pixel in X/Y, and every 5th pixel in Z.
    # This reduces 326 GB down to about ~6.5 Megabytes!
    z_step = 5
    y_step = 100
    x_step = 100
    
    print("Extracting downsampled volume into RAM...")
    # Slicing the memmap pulls ONLY these specific pixels from the hard drive into RAM
    small_vol = volume_map[::z_step, ::y_step, ::x_step]
    
    # Copy to a standard contiguous NumPy array in memory
    small_vol = np.array(small_vol)
    
    # PyVista expects (X, Y, Z) ordering, so we transpose our (Z, Y, X) array
    small_vol = np.transpose(small_vol, (2, 1, 0))
    
    print(f"New downsampled shape: {small_vol.shape}")

    # --- 3. 3D RENDERING ---
    print("Launching PyVista 3D Viewer...")
    grid = pv.ImageData()
    grid.dimensions = np.array(small_vol.shape)
    
    # Because we downsampled differently in Z vs X/Y, the image will look squished.
    # We fix this by telling PyVista the "spacing" between our new voxels.
    # Multiplying Z by your NRRD's physical spacing (0.06) helps maintain real-world proportions.
    grid.spacing = (x_step, y_step, z_step * (20000/815)) 
    
    # Flatten the array for VTK rendering
    grid.point_data["values"] = small_vol.flatten(order="F")

    # Initialize the interactive plotter
    plotter = pv.Plotter(notebook=True)
    
    # Add the volume. 'cmap' changes colors, 'opacity' changes how transparency is mapped
    plotter.add_volume(grid, cmap="bone", opacity="linear")
    
    # Show the viewer in the notebook
    plotter.show(jupyter_backend='trame')